# CrackSpot — Notebook độc lập điều khiển toàn bộ đồ án

Notebook này chứa trực tiếp toàn bộ mã cần để chạy đồ án trên Google Colab hoặc local. Không import package crackspot và không gọi script Python bên ngoài. Repository chỉ cung cấp locked manifest train/validation/test và là nơi lưu phiên bản.

Mục tiêu: phân loại Crack=1 và Non-crack=0 bằng MobileNetV2 224×224, transfer learning E1–E4, tối ưu threshold E5 chỉ trên validation, mở test đúng một lần, giải thích bằng Grad-CAM và đánh giá ảnh tự chụp riêng.

## Đối chiếu yêu cầu

| Nhóm | Nội dung được trình bày trong notebook |
|---|---|
| R01–R03 | Nhãn, đọc RGB/EXIF, giới hạn ảnh, resize và MobileNetV2 preprocess dùng chung |
| R04–R07 | Tải SDNET2018, MD5, giải nén an toàn, locked split 70/15/15 seed 42, kiểm tra leakage |
| R08–R13 | Augmentation chỉ train, class weight từ train, E1/E2/E3, chọn E2/E3 bằng validation, E4, threshold E5 |
| R14–R16 | Test-lock one-shot, Accuracy/Precision/Recall/F1/confusion matrix, TP/TN/FP/FN, Grad-CAM |
| R17–R20 | Một hàm inference chung, upload notebook, latency, ảnh nhóm tự chụp tách riêng |
| R21–R23 | Hash model/manifest, metadata môi trường, report facts, checklist bàn giao |

Lưu ý học thuật: Accuracy ≥ 0,92 là mục tiêu, không phải kết quả có sẵn. Grad-CAM là heatmap vùng mô hình chú ý, không phải mask phân đoạn. Không dùng smoke/validation thay cho kết quả test.


## 0. Cách chạy

1. Colab: chọn Runtime → Change runtime type → GPU.
2. Chỉ sửa cell THAM SỐ ĐIỀU KHIỂN.
3. Lần đầu bật DO_DOWNLOAD_SDNET2018, chạy từ trên xuống và chờ tải khoảng 528 MB.
4. Chạy E1–E3 trước; sau đó E4; sau đó tune E5.
5. Chỉ khi model và threshold đã khóa mới điền FINAL_TEST_TARGETS và chuỗi xác nhận.
6. Ảnh tự chụp không được trộn vào test SDNET2018.

Các cờ mặc định không tự huấn luyện hoặc mở test để tránh tốn GPU và tránh làm sai protocol.


In [ ]:
# ==================== THAM SỐ ĐIỀU KHIỂN ====================
PROJECT_ROOT_OVERRIDE = ""
REPOSITORY_URL = "https://github.com/MKPPF/Do_An_Thi_Giac_May_Tinh.git"
REPOSITORY_REF = "main"
SYNC_REPOSITORY_IN_COLAB = True

# None = tự cài trên Colab, không tự cài local.
INSTALL_DEPENDENCIES = None
DO_DOWNLOAD_SDNET2018 = False
SDNET_ARCHIVE_OVERRIDE = ""  # Có thể trỏ tới ZIP trên Google Drive.
VERIFY_ALL_IMAGE_BYTES = False
SHOW_SPLIT_TABLES = True
SHOW_EDA = True
SHOW_AUGMENTATION_EXAMPLE = False

# Chạy theo giai đoạn để có thể kiểm tra artifact:
# ["E1", "E2", "E3"] trước, sau đó ["E4"].
TRAIN_EXPERIMENTS = []
RUN_IDS = {
    "E1": "e1-official-v1",
    "E2": "e2-official-v1",
    "E3": "e3-official-v1",
    "E4": "e4-official-v1",
}
TUNE_E5 = False

# Test là one-shot. Ví dụ ["E1", "E2", "E3", "E5"].
FINAL_TEST_TARGETS = []
FINAL_TEST_CONFIRMATION = ""
REQUIRED_FINAL_CONFIRMATION = "I_UNDERSTAND_FINAL_TEST_IS_ONE_SHOT"

DO_GRADCAM_GRID = False
DO_BENCHMARK = False
BENCHMARK_IMAGE = "data/external/real/probe.jpg"
DO_REAL_IMAGE_EVALUATION = False
REAL_MANIFEST = "data/external/real/manifest.csv"
DO_NOTEBOOK_DEMO = False
DO_STREAMLIT_DEMO = False  # Tạo app runtime từ chính code trong notebook.
DO_GENERATE_REPORT_FACTS = False
DO_ZIP_ARTIFACTS = False

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
HEAD_EPOCHS = 20
FINE_TUNE_EPOCHS = 30
# =============================================================

## 1. Môi trường, dependency và thư mục dự án


In [ ]:
# ruff: noqa: E402
import importlib.util
import json
import os
import platform
import random
import shlex
import subprocess
import sys
from contextlib import suppress
from pathlib import Path

for stream in (sys.stdout, sys.stderr):
    reconfigure = getattr(stream, "reconfigure", None)
    if reconfigure is not None:
        with suppress(OSError, ValueError):
            reconfigure(encoding="utf-8", errors="replace")

IN_COLAB = importlib.util.find_spec("google.colab") is not None
override = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve() if PROJECT_ROOT_OVERRIDE else None
candidates = [
    p for p in (override, Path.cwd(), Path.cwd() / "CrackSpot", Path("/content/CrackSpot")) if p
]
PROJECT_ROOT = next(
    (p.resolve() for p in candidates if (p / "data/manifests/split_v1").is_dir()), None
)

if PROJECT_ROOT is None and IN_COLAB:
    clone_target = override or Path("/content/CrackSpot")
    if clone_target.exists():
        raise FileExistsError(
            f"{clone_target} tồn tại nhưng không có locked manifest. Hãy xóa runtime."
        )
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            REPOSITORY_REF,
            "--single-branch",
            REPOSITORY_URL,
            str(clone_target),
        ],
        check=True,
    )
    PROJECT_ROOT = clone_target.resolve()

if PROJECT_ROOT is None:
    raise FileNotFoundError("Không tìm thấy repository chứa data/manifests/split_v1.")

if IN_COLAB and SYNC_REPOSITORY_IN_COLAB and (PROJECT_ROOT / ".git").is_dir():
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", REPOSITORY_REF], cwd=PROJECT_ROOT, check=True
    )

os.chdir(PROJECT_ROOT)
PYTHON = sys.executable
python_series = sys.version_info[:2]
if not (3, 10) <= python_series < (3, 14):
    raise RuntimeError(f"Cần Python 3.10-3.13, hiện tại là {sys.version.split()[0]}.")

tensorflow_pin = "2.21.0" if python_series >= (3, 13) else "2.19.0"
h5py_pin = "3.14.0" if python_series >= (3, 13) else "3.16.0"
should_install = IN_COLAB if INSTALL_DEPENDENCIES is None else bool(INSTALL_DEPENDENCIES)
if should_install:
    packages = [
        f"tensorflow=={tensorflow_pin}",
        f"h5py=={h5py_pin}",
        "numpy==2.1.3",
        "pandas==2.2.3",
        "Pillow==11.3.0",
        "scikit-learn==1.6.1",
        "matplotlib==3.10.9",
        "seaborn==0.13.2",
        "ipywidgets>=8.1,<9",
        "streamlit==1.49.1",
    ]
    commands = [
        [
            PYTHON,
            "-m",
            "pip",
            "install",
            "--timeout",
            "120",
            "--upgrade",
            "pip",
            "setuptools",
            "wheel",
        ],
        [PYTHON, "-m", "pip", "install", "--timeout", "120", "--upgrade", *packages],
    ]
    for command in commands:
        print(">>>", shlex.join(command))
        subprocess.run(command, check=True)

import hashlib
import shutil
import statistics
import tempfile
import time
import urllib.request
import zipfile
from http.cookiejar import CookieJar

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf

try:
    from IPython.display import display
except ImportError:

    def display(value):
        print(value)


from PIL import Image, ImageOps
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

if tf.__version__ != tensorflow_pin:
    raise RuntimeError(
        f"TensorFlow={tf.__version__}, cần {tensorflow_pin}. "
        "Nếu vừa cài package, hãy Restart session rồi chạy lại từ đầu."
    )

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception as exc:
    print("Cảnh báo deterministic:", exc)

RAW_ROOT = PROJECT_ROOT / "data/raw"
DATASET_ROOT = RAW_ROOT / "SDNET2018"
ARCHIVE_PATH = RAW_ROOT / "SDNET2018.zip"
SPLIT_ROOT = PROJECT_ROOT / "data/manifests/split_v1"
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts/notebook"
RUN_ROOT = ARTIFACT_ROOT / "runs"
REPORT_ROOT = ARTIFACT_ROOT / "report"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIRS = {name: RUN_ROOT / run_id for name, run_id in RUN_IDS.items()}

git_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, capture_output=True, text=True
).stdout.strip()

print("Project root :", PROJECT_ROOT)
print("Python       :", sys.version.split()[0])
print("TensorFlow   :", tf.__version__)
print("Platform     :", platform.platform())
print("GPU          :", tf.config.list_physical_devices("GPU"))
print("Git commit   :", git_commit)

## 2. Tải, xác minh MD5 và giải nén an toàn SDNET2018


In [ ]:
OFFICIAL_PAGE = "https://digitalcommons.usu.edu/all_datasets/48/"
OFFICIAL_URL = (
    "https://digitalcommons.usu.edu/context/all_datasets/article/1047/type/native/viewcontent"
)
OFFICIAL_MD5 = "677411e784f194422c90f52d9ed0d7c6"
EXPECTED_FOLDERS = ("D/CD", "D/UD", "P/CP", "P/UP", "W/CW", "W/UW")
CHUNK_SIZE = 1024 * 1024


def hash_file(path, algorithm="sha256"):
    digest = hashlib.new(algorithm)
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(CHUNK_SIZE), b""):
            digest.update(chunk)
    return digest.hexdigest()


def download_sdnet(url, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    partial = destination.with_suffix(destination.suffix + ".part")
    offset = partial.stat().st_size if partial.exists() else 0
    opener = urllib.request.build_opener(urllib.request.HTTPCookieProcessor(CookieJar()))
    landing = urllib.request.Request(
        OFFICIAL_PAGE,
        headers={"User-Agent": "Mozilla/5.0", "Accept-Encoding": "identity"},
    )
    with opener.open(landing, timeout=60) as response:
        response.read(1)
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Referer": OFFICIAL_PAGE,
        "Accept-Encoding": "identity",
    }
    if offset:
        headers["Range"] = f"bytes={offset}-"
    with opener.open(urllib.request.Request(url, headers=headers), timeout=120) as response:
        status = getattr(response, "status", response.getcode())
        mode = "ab" if offset and status == 206 else "wb"
        with partial.open(mode) as output:
            while True:
                chunk = response.read(CHUNK_SIZE)
                if not chunk:
                    break
                output.write(chunk)
                print(f"\rĐã tải {output.tell() / 1024**2:,.1f} MB", end="")
    print()
    os.replace(partial, destination)
    return destination


def safe_extract_zip(archive_path, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        members = archive.infolist()
        if len(members) > 100_000:
            raise RuntimeError("ZIP có quá nhiều entry.")
        if sum(item.file_size for item in members) > 20 * 1024**3:
            raise RuntimeError("ZIP vượt giới hạn 20 GB sau giải nén.")
        for item in members:
            relative = Path(item.filename.replace("\\", "/"))
            target = (destination / relative).resolve()
            try:
                target.relative_to(destination)
            except ValueError as exc:
                raise RuntimeError(f"Path traversal trong ZIP: {item.filename}") from exc
            if item.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(item) as source, target.open("wb") as output:
                shutil.copyfileobj(source, output, CHUNK_SIZE)


def find_dataset_root(root):
    root = Path(root)
    candidates = [root] + [p for p in root.rglob("*") if p.is_dir()]
    return next(
        (p for p in candidates if all((p / folder).is_dir() for folder in EXPECTED_FOLDERS)), None
    )


def extract_sdnet_bundle(archive_path, target):
    target = Path(target)
    if target.exists():
        return target
    target.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix="sdnet-", dir=target.parent) as temp_name:
        temp = Path(temp_name)
        outer = temp / "outer"
        safe_extract_zip(archive_path, outer)
        nested = list(outer.rglob("SDNET2018.zip"))
        search_root = outer
        if nested:
            inner = temp / "inner"
            safe_extract_zip(nested[0], inner)
            search_root = inner
        located = find_dataset_root(search_root)
        if located is None:
            raise RuntimeError("Không tìm thấy sáu thư mục lớp SDNET2018 sau giải nén.")
        shutil.move(str(located), str(target))
    return target


if DO_DOWNLOAD_SDNET2018 and not DATASET_ROOT.is_dir():
    if SDNET_ARCHIVE_OVERRIDE:
        source_archive = Path(SDNET_ARCHIVE_OVERRIDE).expanduser()
        if not source_archive.is_file():
            raise FileNotFoundError(source_archive)
        ARCHIVE_PATH.parent.mkdir(parents=True, exist_ok=True)
        if source_archive.resolve() != ARCHIVE_PATH.resolve():
            shutil.copy2(source_archive, ARCHIVE_PATH)
    elif not ARCHIVE_PATH.is_file():
        download_sdnet(OFFICIAL_URL, ARCHIVE_PATH)

    observed_md5 = hash_file(ARCHIVE_PATH, "md5")
    print("Archive MD5:", observed_md5)
    if observed_md5 != OFFICIAL_MD5:
        raise RuntimeError(f"MD5 không khớp: {observed_md5} != {OFFICIAL_MD5}")
    extract_sdnet_bundle(ARCHIVE_PATH, DATASET_ROOT)

print("Dataset:", DATASET_ROOT, "READY" if DATASET_ROOT.is_dir() else "MISSING")

## 3. Hiển thị và kiểm tra toàn bộ train/validation/test

Ba CSV dưới đây chính là phép chia tập; không sao chép 56.088 ảnh sang ba thư mục mới. Cách này tránh nhân ba dữ liệu và vẫn chạy được trên Colab. Notebook kiểm tra hash file, số ảnh, source group, path và exact-image hash không giao nhau.


In [ ]:
EXPECTED_SPLIT_ROWS = {"train": 39014, "validation": 8540, "test": 8534}
EXPECTED_SPLIT_GROUPS = {"train": 160, "validation": 35, "test": 35}
EXPECTED_MANIFEST_SHA = "b38302d62547fb20f264b811f38f7031cf1b8d4c8a21f9db15d4dc39699a3da7"


def read_split(name):
    path = SPLIT_ROOT / f"{name}.csv"
    frame = pd.read_csv(
        path,
        dtype={"relative_path": str, "source_group": str, "sha256": str, "split": str},
        keep_default_na=False,
    )
    required = {"relative_path", "label", "surface", "source_group", "sha256", "split"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{path.name} thiếu cột {sorted(missing)}")
    if set(frame["split"]) != {name}:
        raise ValueError(f"{path.name} có split sai.")
    if len(frame) != EXPECTED_SPLIT_ROWS[name]:
        raise ValueError(f"{name}: sai số dòng {len(frame)}")
    if frame["source_group"].nunique() != EXPECTED_SPLIT_GROUPS[name]:
        raise ValueError(f"{name}: sai số source group.")
    return frame


completion = json.loads((SPLIT_ROOT / "split_complete.json").read_text(encoding="utf-8"))
if completion["manifest_sha256"] != EXPECTED_MANIFEST_SHA:
    raise RuntimeError("Canonical manifest hash không đúng bản locked.")
for filename, expected_hash in completion["artifact_sha256"].items():
    path = SPLIT_ROOT / filename
    if path.is_file() and hash_file(path) != expected_hash:
        raise RuntimeError(f"Artifact locked bị thay đổi bytes: {filename}")

train_df = read_split("train")
val_df = read_split("validation")
test_df = read_split("test")
splits = {"train": train_df, "validation": val_df, "test": test_df}

for left, right in (("train", "validation"), ("train", "test"), ("validation", "test")):
    for column in ("relative_path", "source_group", "sha256"):
        overlap = set(splits[left][column]) & set(splits[right][column])
        if overlap:
            raise RuntimeError(f"LEAKAGE {column} giữa {left}/{right}: {list(overlap)[:3]}")

summary_rows = []
for name, frame in splits.items():
    summary_rows.append(
        {
            "split": name,
            "images": len(frame),
            "source_groups": frame["source_group"].nunique(),
            "crack": int((frame["label"] == 1).sum()),
            "non_crack": int((frame["label"] == 0).sum()),
        }
    )
split_summary = pd.DataFrame(summary_rows)
display(split_summary)

if SHOW_SPLIT_TABLES:
    for name, frame in splits.items():
        print(f"\n=== {name.upper()} — {len(frame):,} ảnh ===")
        display(frame.groupby(["surface", "label"]).size().rename("images").reset_index())
        display(
            frame[
                ["relative_path", "label", "class_name", "surface", "source_group", "sha256"]
            ].head(10)
        )
        print("CSV đầy đủ:", SPLIT_ROOT / f"{name}.csv")

if DATASET_ROOT.is_dir():
    missing = []
    for frame in splits.values():
        for relative in frame["relative_path"]:
            if not (DATASET_ROOT / relative).is_file():
                missing.append(relative)
                if len(missing) >= 10:
                    break
    if missing:
        raise FileNotFoundError(f"Thiếu ảnh dataset: {missing}")

    if VERIFY_ALL_IMAGE_BYTES:
        checked = 0
        for frame in splits.values():
            for row in frame.itertuples(index=False):
                path = DATASET_ROOT / row.relative_path
                if hash_file(path) != row.sha256:
                    raise RuntimeError(f"SHA-256 ảnh không khớp: {row.relative_path}")
                checked += 1
                if checked % 2000 == 0:
                    print(f"\rĐã xác minh {checked:,}/56.088 ảnh", end="")
        print(f"\nFULL BYTE CHECK PASS: {checked:,} ảnh")

if SHOW_EDA:
    figure, axes = plt.subplots(1, 2, figsize=(13, 4))
    sns.barplot(
        data=split_summary.melt("split", ["crack", "non_crack"]),
        x="split",
        y="value",
        hue="variable",
        ax=axes[0],
    )
    axes[0].set_title("Phân bố nhãn theo split")
    surface_counts = (
        pd.concat(splits.values()).groupby(["split", "surface"]).size().reset_index(name="images")
    )
    sns.barplot(data=surface_counts, x="split", y="images", hue="surface", ax=axes[1])
    axes[1].set_title("Phân bố bề mặt D/P/W")
    plt.tight_layout()
    plt.show()

print("LOCKED SPLIT PASS — zero path/group/hash leakage.")

## 4. Đọc ảnh, augmentation, tf.data, class weight và MobileNetV2


In [ ]:
MAX_IMAGE_BYTES = 10 * 1024**2
MAX_IMAGE_PIXELS = 25_000_000


def load_rgb_array(path):
    path = Path(path)
    if path.stat().st_size > MAX_IMAGE_BYTES:
        raise ValueError(f"Ảnh vượt {MAX_IMAGE_BYTES} bytes: {path}")
    with Image.open(path) as image:
        image = ImageOps.exif_transpose(image)
        if image.width * image.height > MAX_IMAGE_PIXELS:
            raise ValueError(f"Ảnh có quá nhiều pixel: {path}")
        return np.asarray(image.convert("RGB"), dtype=np.uint8)


augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal", seed=SEED),
        tf.keras.layers.RandomRotation(15 / 360, fill_mode="reflect", seed=SEED + 1),
        tf.keras.layers.RandomBrightness(0.15, value_range=(0, 255), seed=SEED + 2),
        tf.keras.layers.RandomContrast(0.15, seed=SEED + 3),
    ],
    name="train_only_augmentation",
)


def build_dataset(frame, training=False, augment=False, include_paths=False):
    if augment and not training:
        raise ValueError("Augmentation chỉ được phép trên train.")
    paths = np.asarray([str(DATASET_ROOT / value) for value in frame["relative_path"]])
    labels = frame["label"].to_numpy(np.float32)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    options = tf.data.Options()
    options.experimental_deterministic = True
    ds = ds.with_options(options)
    if training:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)

    def decode(path, label):
        def pil_loader(raw):
            value = raw.numpy().decode("utf-8")
            return load_rgb_array(value)

        image = tf.py_function(pil_loader, [path], Tout=tf.uint8)
        image.set_shape((None, None, 3))
        image = tf.image.resize(tf.cast(image, tf.float32), IMAGE_SIZE, antialias=True)
        image.set_shape((*IMAGE_SIZE, 3))
        if augment:
            image = augmentation(image, training=True)
            image = tf.clip_by_value(image, 0.0, 255.0)
        image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
        if include_paths:
            return image, label, path
        return image, label

    ds = ds.map(decode, num_parallel_calls=tf.data.AUTOTUNE, deterministic=True)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


def balanced_class_weights(frame):
    counts = frame["label"].value_counts().to_dict()
    total = len(frame)
    return {label: total / (2.0 * counts[label]) for label in (0, 1)}


CLASS_WEIGHTS = balanced_class_weights(train_df)
print("Class weights chỉ từ train:", CLASS_WEIGHTS)


def build_model():
    inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3), name="image")
    backbone = tf.keras.applications.MobileNetV2(
        input_shape=(*IMAGE_SIZE, 3), include_top=False, weights="imagenet"
    )
    backbone.trainable = False
    features = backbone(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pooling")(features)
    x = tf.keras.layers.Dropout(0.30, name="classifier_dropout")(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid", name="crack_probability")(x)
    return tf.keras.Model(inputs, outputs, name="crackspot_mobilenetv2")


def find_backbone(model):
    candidates = [
        layer
        for layer in model.layers
        if isinstance(layer, tf.keras.Model)
        and "mobilenetv2" in layer.name.replace("_", "").lower()
    ]
    if len(candidates) != 1:
        raise ValueError(f"Cần đúng một MobileNetV2 backbone, tìm thấy {len(candidates)}")
    return candidates[0]


def configure_fine_tuning(model, boundary=None):
    backbone = find_backbone(model)
    if boundary is None:
        backbone.trainable = False
    else:
        names = [layer.name for layer in backbone.layers]
        if boundary not in names:
            raise ValueError(f"Không tìm thấy layer {boundary}")
        start = names.index(boundary)
        backbone.trainable = True
        for index, layer in enumerate(backbone.layers):
            layer.trainable = index >= start and not isinstance(
                layer, tf.keras.layers.BatchNormalization
            )
    return {
        "boundary": boundary,
        "total_parameters": int(model.count_params()),
        "trainable_parameters": int(sum(np.prod(v.shape) for v in model.trainable_weights)),
        "trainable_backbone_layers": [layer.name for layer in backbone.layers if layer.trainable],
        "trainable_batch_norm_layers": [
            layer.name
            for layer in backbone.layers
            if isinstance(layer, tf.keras.layers.BatchNormalization) and layer.trainable
        ],
    }


def compile_model(model, learning_rate):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate),
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=[tf.keras.metrics.BinaryAccuracy(name="accuracy")],
    )
    return model


if SHOW_AUGMENTATION_EXAMPLE and DATASET_ROOT.is_dir():
    sample_path = DATASET_ROOT / train_df.iloc[0]["relative_path"]
    original = load_rgb_array(sample_path)
    resized = tf.image.resize(original, IMAGE_SIZE)
    augmented = augmentation(resized, training=True).numpy().astype(np.uint8)
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    axes[0].imshow(original)
    axes[0].set_title("Gốc")
    axes[1].imshow(augmented)
    axes[1].set_title("Augmentation train E4")
    for axis in axes:
        axis.axis("off")
    plt.tight_layout()
    plt.show()

## 5. Huấn luyện E1–E4

- E1: backbone frozen, head LR 1e-3, không augmentation.
- E2: head rồi unfreeze từ block_14_expand, LR 1e-4, BatchNorm frozen.
- E3: head rồi unfreeze từ block_10_expand, LR 1e-5, BatchNorm frozen.
- E4: chọn E2/E3 chỉ bằng best validation loss, sau đó huấn luyện lại với augmentation.


In [ ]:
EXPERIMENTS = {
    "E1": {"boundary": None, "fine_lr": None, "augment": False},
    "E2": {"boundary": "block_14_expand", "fine_lr": 1e-4, "augment": False},
    "E3": {"boundary": "block_10_expand", "fine_lr": 1e-5, "augment": False},
}


def json_default(value):
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(type(value).__name__)


def write_json(path, payload):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(
        json.dumps(payload, ensure_ascii=False, indent=2, default=json_default),
        encoding="utf-8",
    )


def compute_binary_metrics(y_true, probabilities, threshold=0.5):
    truth = np.asarray(y_true, dtype=np.int64)
    scores = np.asarray(probabilities, dtype=np.float64)
    predicted = (scores >= threshold).astype(np.int64)
    precision, recall, f1, support = precision_recall_fscore_support(
        truth, predicted, labels=[0, 1], zero_division=0
    )
    macro = precision_recall_fscore_support(truth, predicted, average="macro", zero_division=0)
    matrix = confusion_matrix(truth, predicted, labels=[0, 1])
    tn, fp, fn, tp = map(int, matrix.ravel())
    return {
        "threshold": float(threshold),
        "sample_count": len(truth),
        "accuracy": float(accuracy_score(truth, predicted)),
        "non_crack": {
            "precision": float(precision[0]),
            "recall": float(recall[0]),
            "f1": float(f1[0]),
            "support": int(support[0]),
        },
        "crack": {
            "precision": float(precision[1]),
            "recall": float(recall[1]),
            "f1": float(f1[1]),
            "support": int(support[1]),
        },
        "macro": {"precision": float(macro[0]), "recall": float(macro[1]), "f1": float(macro[2])},
        "confusion_matrix": matrix.tolist(),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "classification_report": classification_report(
            truth,
            predicted,
            labels=[0, 1],
            target_names=["Non-crack", "Crack"],
            output_dict=True,
            zero_division=0,
        ),
    }


def predict_frame(model, frame):
    dataset = build_dataset(frame, training=False, augment=False)
    probabilities = model.predict(dataset, verbose=1).reshape(-1)
    result = frame[["relative_path", "label", "surface", "source_group"]].copy()
    result = result.rename(columns={"label": "y_true"})
    result["probability_crack"] = probabilities
    return result


def callbacks_for(checkpoint):
    return [
        tf.keras.callbacks.ModelCheckpoint(
            checkpoint, monitor="val_loss", mode="min", save_best_only=True, verbose=1
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", mode="min", patience=5, restore_best_weights=True, verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", mode="min", factor=0.2, patience=2, min_lr=1e-7, verbose=1
        ),
        tf.keras.callbacks.CSVLogger(str(Path(checkpoint).with_suffix(".csv"))),
    ]


def load_run_summary(experiment):
    path = RUN_DIRS[experiment] / "run_summary.json"
    if not path.is_file():
        raise FileNotFoundError(f"Chưa có run hoàn chỉnh {experiment}: {path}")
    return json.loads(path.read_text(encoding="utf-8"))


def train_experiment(experiment, config):
    run_dir = RUN_DIRS[experiment]
    completed = run_dir / "run_summary.json"
    if completed.is_file():
        print(f"{experiment}: đã hoàn tất, không ghi đè {run_dir}")
        return json.loads(completed.read_text(encoding="utf-8"))
    if run_dir.exists():
        raise FileExistsError(
            f"{run_dir} tồn tại nhưng chưa hoàn tất. Hãy đổi RUN_IDS để không ghi đè bằng chứng."
        )
    if not DATASET_ROOT.is_dir():
        raise FileNotFoundError("Thiếu SDNET2018. Bật DO_DOWNLOAD_SDNET2018.")
    run_dir.mkdir(parents=True)

    train_ds = build_dataset(train_df, training=True, augment=config["augment"])
    val_ds = build_dataset(val_df, training=False)
    model = build_model()
    with (run_dir / "model_summary.txt").open("w", encoding="utf-8") as summary_file:
        model.summary(print_fn=lambda line: summary_file.write(line + "\n"))
    trainability_head = configure_fine_tuning(model, None)
    compile_model(model, 1e-3)

    started = time.perf_counter()
    head_checkpoint = run_dir / "head_best.keras"
    head_history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=HEAD_EPOCHS,
        class_weight=CLASS_WEIGHTS,
        callbacks=callbacks_for(head_checkpoint),
        verbose=1,
    )
    histories = {"head": head_history.history}
    candidates = [(min(head_history.history["val_loss"]), head_checkpoint, "head")]

    boundary = config.get("boundary")
    if boundary:
        model = tf.keras.models.load_model(head_checkpoint)
        trainability_fine = configure_fine_tuning(model, boundary)
        if trainability_fine["trainable_batch_norm_layers"]:
            raise RuntimeError("BatchNormalization phải frozen khi fine-tune.")
        compile_model(model, config["fine_lr"])
        fine_checkpoint = run_dir / "fine_best.keras"
        fine_history = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=FINE_TUNE_EPOCHS,
            class_weight=CLASS_WEIGHTS,
            callbacks=callbacks_for(fine_checkpoint),
            verbose=1,
        )
        histories["fine_tune"] = fine_history.history
        candidates.append((min(fine_history.history["val_loss"]), fine_checkpoint, "fine_tune"))
    else:
        trainability_fine = trainability_head

    best_val_loss, best_checkpoint, best_phase = min(candidates, key=lambda item: item[0])
    final_model_path = run_dir / "model.keras"
    shutil.copy2(best_checkpoint, final_model_path)
    model = tf.keras.models.load_model(final_model_path)
    predictions = predict_frame(model, val_df)
    metrics = compute_binary_metrics(
        predictions["y_true"], predictions["probability_crack"], threshold=0.5
    )
    duration = time.perf_counter() - started
    model_sha = hash_file(final_model_path)

    predictions.assign(split="validation").to_csv(
        run_dir / "predictions_validation.csv", index=False
    )
    write_json(run_dir / "history.json", histories)
    write_json(run_dir / "metrics_validation.json", metrics)
    write_json(run_dir / "trainability.json", trainability_fine)
    metadata = {
        "experiment": experiment,
        "run_id": RUN_IDS[experiment],
        "model_sha256": model_sha,
        "manifest_sha256": EXPECTED_MANIFEST_SHA,
        "git_commit": git_commit,
        "tensorflow_version": tf.__version__,
        "input_size": list(IMAGE_SIZE),
        "preprocessing": "mobilenet_v2.preprocess_input",
        "label_mapping": {"Non-crack": 0, "Crack": 1},
        "threshold": 0.5,
        "gradcam_layer": "out_relu",
        "best_phase": best_phase,
        "best_val_loss": float(best_val_loss),
        "duration_seconds": duration,
        "config": config,
    }
    write_json(run_dir / "model.metadata.json", metadata)
    summary = {
        **metadata,
        "status": "VALIDATION_COMPLETE_TEST_LOCKED",
        "valid_for_report": True,
        "metrics_validation": metrics,
    }
    write_json(completed, summary)
    print(f"{experiment} DONE — val_loss={best_val_loss:.6f}, test vẫn LOCKED")
    tf.keras.backend.clear_session()
    return summary

In [ ]:
requested = [name.upper() for name in TRAIN_EXPERIMENTS]
if len(requested) != len(set(requested)) or not set(requested).issubset({"E1", "E2", "E3", "E4"}):
    raise ValueError("TRAIN_EXPERIMENTS chỉ nhận E1/E2/E3/E4, không lặp.")

for experiment in ("E1", "E2", "E3"):
    if experiment in requested:
        train_experiment(experiment, EXPERIMENTS[experiment])

if "E4" in requested:
    e2 = load_run_summary("E2")
    e3 = load_run_summary("E3")
    candidates = [e2, e3]
    winner = min(candidates, key=lambda item: (item["best_val_loss"], item["experiment"]))
    selection = {
        "selection_split": "validation",
        "primary_metric": "best_val_loss",
        "forbid_test_access": True,
        "winner": winner["experiment"],
        "winner_run_id": winner["run_id"],
        "winner_model_sha256": winner["model_sha256"],
        "candidates": [
            {
                "experiment": item["experiment"],
                "run_id": item["run_id"],
                "best_val_loss": item["best_val_loss"],
                "model_sha256": item["model_sha256"],
            }
            for item in candidates
        ],
    }
    write_json(ARTIFACT_ROOT / "model_selection.json", selection)
    winner_config = EXPERIMENTS[winner["experiment"]]
    e4_config = {
        "boundary": winner_config["boundary"],
        "fine_lr": winner_config["fine_lr"],
        "augment": True,
        "inherited_from": winner["experiment"],
        "selection_artifact": str(ARTIFACT_ROOT / "model_selection.json"),
    }
    train_experiment("E4", e4_config)

if not requested:
    print("TRAINING LOCKED: TRAIN_EXPERIMENTS đang rỗng.")

## 6. E5: tối ưu threshold chỉ trên validation


In [ ]:
def optimize_threshold(y_true, probabilities):
    truth = np.asarray(y_true, dtype=np.int64)
    scores = np.asarray(probabilities, dtype=np.float64)
    order = np.argsort(scores)
    sorted_scores = scores[order]
    sorted_truth = truth[order]
    prefix_positive = np.concatenate([[0], np.cumsum(sorted_truth)])
    total_positive = int(sorted_truth.sum())
    candidates = np.unique(np.concatenate([[0.0, 0.5, 1.0], scores]))
    ranked = []
    for threshold in candidates:
        index = int(np.searchsorted(sorted_scores, threshold, side="left"))
        tp = total_positive - int(prefix_positive[index])
        predicted_positive = len(scores) - index
        fp = predicted_positive - tp
        fn = total_positive - tp
        recall = tp / (tp + fn) if tp + fn else 0.0
        precision = tp / (tp + fp) if tp + fp else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        key = (f1, recall, -abs(float(threshold) - 0.5), -float(threshold))
        ranked.append((key, float(threshold), precision, recall, f1))
    _, threshold, precision, recall, f1 = max(ranked, key=lambda item: item[0])
    return {
        "threshold": threshold,
        "precision_crack": precision,
        "recall_crack": recall,
        "f1_crack": f1,
        "evaluated_candidates": len(candidates),
        "tie_break": ["f1 desc", "recall desc", "distance to 0.5 asc", "threshold asc"],
        "source_split": "validation",
    }


THRESHOLD_PATH = RUN_DIRS["E4"] / "threshold_validation.json"
if TUNE_E5:
    if THRESHOLD_PATH.exists():
        raise FileExistsError(f"Threshold E5 đã khóa; không ghi đè: {THRESHOLD_PATH}")
    e4_summary = load_run_summary("E4")
    predictions_path = RUN_DIRS["E4"] / "predictions_validation.csv"
    predictions = pd.read_csv(predictions_path)
    if set(predictions.get("split", pd.Series(["validation"])).astype(str)) != {"validation"}:
        raise RuntimeError("E5 chỉ được tune từ validation.")
    threshold_result = optimize_threshold(predictions["y_true"], predictions["probability_crack"])
    threshold_result.update(
        {
            "experiment": "E5",
            "is_new_model": False,
            "model_source": "E4",
            "model_sha256": e4_summary["model_sha256"],
            "manifest_sha256": EXPECTED_MANIFEST_SHA,
            "predictions_validation_sha256": hash_file(predictions_path),
            "status": "VALIDATION_THRESHOLD_LOCKED_TEST_UNSEEN",
        }
    )
    write_json(THRESHOLD_PATH, threshold_result)
    print(json.dumps(threshold_result, ensure_ascii=False, indent=2))
elif THRESHOLD_PATH.is_file():
    threshold_result = json.loads(THRESHOLD_PATH.read_text(encoding="utf-8"))
    print("E5 đã khóa:", threshold_result["threshold"])
else:
    threshold_result = None
    print("E5 LOCKED: đặt TUNE_E5=True sau khi E4 hoàn tất.")

## 7. Final test one-shot

Cell này là nơi duy nhất đọc test để tính metric. Mỗi SHA-256 checkpoint chỉ được đánh giá một lần và được ghi vào registry. Không chạy cell với target cho đến khi E1–E4 và E5 đã khóa.


In [ ]:
FINAL_REGISTRY = ARTIFACT_ROOT / "final_test_registry.json"


def evaluate_model_on_frame(model_path, frame, threshold):
    model = tf.keras.models.load_model(model_path)
    predictions = predict_frame(model, frame)
    metrics = compute_binary_metrics(
        predictions["y_true"], predictions["probability_crack"], threshold
    )
    predictions["y_pred"] = (predictions["probability_crack"].to_numpy() >= threshold).astype(int)
    return metrics, predictions


registry = (
    json.loads(FINAL_REGISTRY.read_text(encoding="utf-8")) if FINAL_REGISTRY.is_file() else {}
)


def final_target_readiness(target):
    source = "E4" if target == "E5" else target
    run_dir = RUN_DIRS[source]
    required = [run_dir / "run_summary.json", run_dir / "model.keras"]
    if target == "E5":
        required.append(THRESHOLD_PATH)
    missing = [str(path.relative_to(PROJECT_ROOT)) for path in required if not path.is_file()]
    if missing:
        return {"target": target, "status": "MISSING", "detail": "; ".join(missing)}
    try:
        summary = json.loads((run_dir / "run_summary.json").read_text(encoding="utf-8"))
        model_sha = hash_file(run_dir / "model.keras")
        if model_sha != summary.get("model_sha256"):
            return {"target": target, "status": "HASH_MISMATCH", "detail": model_sha}
        if target == "E5":
            locked = json.loads(THRESHOLD_PATH.read_text(encoding="utf-8"))
            if locked.get("model_sha256") != model_sha:
                return {"target": target, "status": "THRESHOLD_MISMATCH", "detail": model_sha}
        if model_sha in registry:
            return {"target": target, "status": "ALREADY_EVALUATED", "detail": model_sha}
        return {"target": target, "status": "READY", "detail": model_sha}
    except (OSError, ValueError, KeyError, json.JSONDecodeError) as exc:
        return {"target": target, "status": "INVALID_ARTIFACT", "detail": str(exc)}


readiness_rows = [final_target_readiness(target) for target in ("E1", "E2", "E3", "E5")]
readiness = pd.DataFrame(readiness_rows)
display(readiness)
readiness_by_target = {row["target"]: row["status"] for row in readiness_rows}

targets = [name.upper() for name in FINAL_TEST_TARGETS]
if targets:
    if FINAL_TEST_CONFIRMATION != REQUIRED_FINAL_CONFIRMATION:
        raise RuntimeError("Sai FINAL_TEST_CONFIRMATION; test vẫn khóa.")
    if len(targets) != len(set(targets)) or not set(targets).issubset({"E1", "E2", "E3", "E5"}):
        raise ValueError("FINAL_TEST_TARGETS chỉ nhận E1/E2/E3/E5, không lặp.")

    not_ready = {
        target: readiness_by_target[target]
        for target in targets
        if readiness_by_target[target] != "READY"
    }
    if not_ready:
        raise RuntimeError(f"Final test chưa sẵn sàng: {not_ready}")
    for target in targets:
        source = "E4" if target == "E5" else target
        summary = load_run_summary(source)
        model_path = RUN_DIRS[source] / "model.keras"
        model_sha = hash_file(model_path)
        if model_sha != summary["model_sha256"]:
            raise RuntimeError(f"Checkpoint {source} đã thay đổi hash.")
        if model_sha in registry:
            raise RuntimeError(f"Checkpoint {model_sha[:12]} đã final-test; từ chối chạy lại.")
        if target == "E5":
            if not THRESHOLD_PATH.is_file():
                raise FileNotFoundError("Chưa khóa threshold E5.")
            locked = json.loads(THRESHOLD_PATH.read_text(encoding="utf-8"))
            if locked["model_sha256"] != model_sha:
                raise RuntimeError("Threshold E5 không thuộc checkpoint E4 hiện tại.")
            threshold = float(locked["threshold"])
        else:
            threshold = 0.5

        output_dir = REPORT_ROOT / "final_evaluation" / target.lower()
        if output_dir.exists():
            raise FileExistsError(f"Không ghi đè final evidence: {output_dir}")
        output_dir.mkdir(parents=True)
        metrics, predictions = evaluate_model_on_frame(model_path, test_df, threshold)
        predictions.assign(split="test").to_csv(output_dir / "predictions_test.csv", index=False)
        payload = {
            "experiment": target,
            "model_source": source,
            "threshold": threshold,
            "model_sha256": model_sha,
            "manifest_sha256": EXPECTED_MANIFEST_SHA,
            "git_commit": git_commit,
            "metrics": metrics,
            "accuracy_target_0_92_met": bool(metrics["accuracy"] >= 0.92),
            "status": "FINAL_TEST_COMPLETE_ONE_SHOT",
        }
        if target == "E5":
            payload["metrics_same_checkpoint_threshold_0_5"] = compute_binary_metrics(
                predictions["y_true"], predictions["probability_crack"], 0.5
            )
        write_json(output_dir / "metrics_test.json", payload)
        registry[model_sha] = {
            "experiment": target,
            "output": str(output_dir),
            "metrics_sha256": hash_file(output_dir / "metrics_test.json"),
        }
        write_json(FINAL_REGISTRY, registry)
        print(target, json.dumps(metrics, ensure_ascii=False, indent=2))
else:
    ready = [row["target"] for row in readiness_rows if row["status"] == "READY"]
    print("FINAL TEST LOCKED an toàn — đây không phải lỗi.")
    if ready:
        print("Target đã đủ artifact:", ready)
        print("Khi quyết định mở test, đặt trong cell tham số:")
        print(f"FINAL_TEST_TARGETS = {ready!r}")
        print(f'FINAL_TEST_CONFIRMATION = "{REQUIRED_FINAL_CONFIRMATION}"')
    else:
        print("Chưa có target READY. Hãy hoàn tất E1-E3, E4 và TUNE_E5 theo bảng trên.")

## 8. Learning curves, confusion matrix và Grad-CAM TP/TN/FP/FN


In [ ]:
def plot_training_history(run_dir, title):
    payload = json.loads((Path(run_dir) / "history.json").read_text(encoding="utf-8"))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    offset = 0
    for phase, history in payload.items():
        epochs = np.arange(1, len(history["loss"]) + 1) + offset
        axes[0].plot(epochs, history["loss"], label=f"{phase} train")
        axes[0].plot(epochs, history["val_loss"], "--", label=f"{phase} val")
        axes[1].plot(epochs, history["accuracy"], label=f"{phase} train")
        axes[1].plot(epochs, history["val_accuracy"], "--", label=f"{phase} val")
        offset = int(epochs[-1])
    axes[0].set_title(f"{title} — Loss")
    axes[1].set_title(f"{title} — Accuracy")
    for axis in axes:
        axis.set_xlabel("Epoch")
        axis.legend()
        axis.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()


for experiment in ("E1", "E2", "E3", "E4"):
    if (RUN_DIRS[experiment] / "history.json").is_file():
        plot_training_history(RUN_DIRS[experiment], experiment)


def show_confusion(payload, title):
    matrix = np.asarray(payload["metrics"]["confusion_matrix"])
    sns.heatmap(
        matrix,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Non-crack", "Crack"],
        yticklabels=["Non-crack", "Crack"],
    )
    plt.xlabel("Dự đoán")
    plt.ylabel("Thực tế")
    plt.title(title)
    plt.show()


for target in ("e1", "e2", "e3", "e5"):
    path = REPORT_ROOT / "final_evaluation" / target / "metrics_test.json"
    if path.is_file():
        show_confusion(
            json.loads(path.read_text(encoding="utf-8")), f"Confusion matrix {target.upper()}"
        )


def preprocess_single_image(path):
    original = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    resized = original.resize(IMAGE_SIZE[::-1], Image.Resampling.BILINEAR)
    array = np.asarray(resized, dtype=np.float32)
    batch = tf.keras.applications.mobilenet_v2.preprocess_input(array)[None, ...]
    return original, batch


def make_gradcam_heatmap(model, batch, layer_name="out_relu"):
    backbone = find_backbone(model)
    target = backbone.get_layer(layer_name)
    feature_model = tf.keras.Model(backbone.input, [target.output, backbone.output])
    pooling = model.get_layer("global_average_pooling")
    dropout = model.get_layer("classifier_dropout")
    classifier = model.get_layer("crack_probability")
    with tf.GradientTape() as tape:
        conv, features = feature_model(batch, training=False)
        score = classifier(dropout(pooling(features), training=False), training=False)[0, 0]
    gradients = tape.gradient(score, conv)
    weights = tf.reduce_mean(gradients, axis=(1, 2), keepdims=True)
    heatmap = tf.nn.relu(tf.reduce_sum(weights * conv, axis=-1))[0].numpy()
    maximum = float(heatmap.max())
    return heatmap / maximum if maximum > 1e-12 else np.zeros_like(heatmap)


def overlay_gradcam(original, heatmap, alpha=0.40):
    heat = Image.fromarray(np.uint8(np.clip(heatmap, 0, 1) * 255)).resize(original.size)
    color = plt.get_cmap("turbo")(np.asarray(heat) / 255.0)[..., :3]
    base = np.asarray(original, dtype=np.float32) / 255.0
    overlay = np.clip((1 - alpha) * base + alpha * color, 0, 1)
    return Image.fromarray(np.uint8(overlay * 255))


if DO_GRADCAM_GRID:
    prediction_path = REPORT_ROOT / "final_evaluation/e5/predictions_test.csv"
    if not prediction_path.is_file():
        raise FileNotFoundError("Cần final evaluation E5 trước khi sinh Grad-CAM grid.")
    pred = pd.read_csv(prediction_path)
    pred["case"] = np.select(
        [
            (pred.y_true == 1) & (pred.y_pred == 1),
            (pred.y_true == 0) & (pred.y_pred == 0),
            (pred.y_true == 0) & (pred.y_pred == 1),
            (pred.y_true == 1) & (pred.y_pred == 0),
        ],
        ["TP", "TN", "FP", "FN"],
        default="",
    )
    model = tf.keras.models.load_model(RUN_DIRS["E4"] / "model.keras")
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    for column, case in enumerate(("TP", "TN", "FP", "FN")):
        rows = pred[pred["case"] == case]
        if rows.empty:
            axes[0, column].text(0.5, 0.5, f"Không có {case}", ha="center")
            axes[1, column].axis("off")
            continue
        row = rows.sort_values("probability_crack", ascending=(case in {"TN", "FN"})).iloc[0]
        original, batch = preprocess_single_image(DATASET_ROOT / row.relative_path)
        heatmap = make_gradcam_heatmap(model, batch)
        overlay = overlay_gradcam(original, heatmap)
        axes[0, column].imshow(original)
        axes[0, column].set_title(f"{case} gốc\nP={row.probability_crack:.3f}")
        axes[1, column].imshow(overlay)
        axes[1, column].set_title(f"{case} Grad-CAM")
        axes[0, column].axis("off")
        axes[1, column].axis("off")
    plt.tight_layout()
    output = REPORT_ROOT / "gradcam_tp_tn_fp_fn.png"
    fig.savefig(output, dpi=200, bbox_inches="tight")
    plt.show()
    print("Đã lưu:", output)

## 9. Inference chung, latency, ảnh tự chụp và upload demo


In [ ]:
def predict_image(model, image_path, threshold):
    original, batch = preprocess_single_image(image_path)
    started = time.perf_counter()
    probability = float(model.predict(batch, verbose=0)[0, 0])
    latency = time.perf_counter() - started
    heatmap = make_gradcam_heatmap(model, batch)
    return {
        "label": "Crack" if probability >= threshold else "Non-crack",
        "probability_crack": probability,
        "threshold": float(threshold),
        "latency_seconds": latency,
        "original": original,
        "gradcam": overlay_gradcam(original, heatmap),
    }


def selected_e5_model_and_threshold():
    if not THRESHOLD_PATH.is_file():
        raise FileNotFoundError("Chưa có threshold E5.")
    locked = json.loads(THRESHOLD_PATH.read_text(encoding="utf-8"))
    model_path = RUN_DIRS["E4"] / "model.keras"
    if hash_file(model_path) != locked["model_sha256"]:
        raise RuntimeError("Model E4 không khớp threshold E5.")
    return tf.keras.models.load_model(model_path), float(locked["threshold"])


if DO_BENCHMARK:
    benchmark_path = PROJECT_ROOT / BENCHMARK_IMAGE
    if not benchmark_path.is_file():
        raise FileNotFoundError("BENCHMARK_IMAGE phải là ảnh ngoài test SDNET2018.")
    model, threshold = selected_e5_model_and_threshold()
    _, batch = preprocess_single_image(benchmark_path)
    for _ in range(10):
        model.predict(batch, verbose=0)
    samples = []
    for _ in range(100):
        start = time.perf_counter()
        model.predict(batch, verbose=0)
        samples.append(time.perf_counter() - start)
    benchmark = {
        "image": str(benchmark_path),
        "warmup_runs": 10,
        "measured_runs": 100,
        "mean_seconds": statistics.mean(samples),
        "median_seconds": statistics.median(samples),
        "p95_seconds": float(np.percentile(samples, 95)),
        "target_seconds": 5.0,
        "target_met": bool(np.percentile(samples, 95) <= 5.0),
        "tensorflow_version": tf.__version__,
        "device": [str(item) for item in tf.config.list_physical_devices()],
    }
    write_json(REPORT_ROOT / "benchmark.json", benchmark)
    display(pd.DataFrame([benchmark]))


if DO_REAL_IMAGE_EVALUATION:
    manifest_path = PROJECT_ROOT / REAL_MANIFEST
    real = pd.read_csv(manifest_path)
    required = {"relative_path", "label", "capture_source"}
    if not required.issubset(real.columns):
        raise ValueError(f"Real manifest thiếu {sorted(required - set(real.columns))}")
    if set(real["capture_source"]) != {"self_captured"}:
        raise ValueError("Mọi ảnh phải khai báo capture_source=self_captured.")
    if set(real["relative_path"]) & set(test_df["relative_path"]):
        raise RuntimeError("Ảnh tự chụp không được trộn với test SDNET2018.")
    model, threshold = selected_e5_model_and_threshold()
    root = manifest_path.parent
    probabilities = []
    for relative in real["relative_path"]:
        result = predict_image(model, root / relative, threshold)
        probabilities.append(result["probability_crack"])
    real["probability_crack"] = probabilities
    real_metrics = compute_binary_metrics(real["label"], probabilities, threshold)
    output = REPORT_ROOT / "real_images"
    output.mkdir(parents=True, exist_ok=True)
    real.to_csv(output / "predictions_real.csv", index=False)
    write_json(output / "metrics_real.json", real_metrics)
    print(json.dumps(real_metrics, ensure_ascii=False, indent=2))


if DO_NOTEBOOK_DEMO:
    model, threshold = selected_e5_model_and_threshold()
    if IN_COLAB:
        from google.colab import files

        uploaded = files.upload()
        demo_paths = []
        for filename, content in uploaded.items():
            target = ARTIFACT_ROOT / "uploads" / Path(filename).name
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(content)
            demo_paths.append(target)
    else:
        demo_paths = [PROJECT_ROOT / BENCHMARK_IMAGE]

    for path in demo_paths:
        result = predict_image(model, path, threshold)
        print(
            f"{path.name}: {result['label']} | P(Crack)={result['probability_crack']:.4f} "
            f"| threshold={threshold:.4f} | latency={result['latency_seconds']:.3f}s"
        )
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        axes[0].imshow(result["original"])
        axes[0].set_title("Ảnh gốc")
        axes[1].imshow(result["gradcam"])
        axes[1].set_title("Grad-CAM — vùng chú ý")
        for axis in axes:
            axis.axis("off")
        plt.tight_layout()
        plt.show()

## 10. Streamlit demo được sinh trực tiếp từ notebook

Mã app nằm trọn trong cell dưới. File runtime chỉ được tạo khi bật cờ, không phải source phụ của đồ án.


In [ ]:
STREAMLIT_APP_SOURCE = r"""
import json
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import streamlit as st
import tensorflow as tf
from PIL import Image, ImageOps

MODEL_PATH = Path(os.environ["CRACKSPOT_MODEL_PATH"])
THRESHOLD_PATH = Path(os.environ["CRACKSPOT_THRESHOLD_PATH"])
IMAGE_SIZE = (224, 224)

@st.cache_resource
def load_assets():
    metadata = json.loads(THRESHOLD_PATH.read_text(encoding="utf-8"))
    return tf.keras.models.load_model(MODEL_PATH), float(metadata["threshold"])

def find_backbone(model):
    items = [layer for layer in model.layers if isinstance(layer, tf.keras.Model)
             and "mobilenetv2" in layer.name.replace("_", "").lower()]
    if len(items) != 1:
        raise ValueError("Không tìm thấy duy nhất MobileNetV2 backbone.")
    return items[0]

def preprocess(image):
    image = ImageOps.exif_transpose(image).convert("RGB")
    resized = image.resize(IMAGE_SIZE[::-1], Image.Resampling.BILINEAR)
    array = np.asarray(resized, dtype=np.float32)
    batch = tf.keras.applications.mobilenet_v2.preprocess_input(array)[None, ...]
    return image, batch

def gradcam(model, batch):
    backbone = find_backbone(model)
    feature_model = tf.keras.Model(
        backbone.input, [backbone.get_layer("out_relu").output, backbone.output]
    )
    with tf.GradientTape() as tape:
        conv, features = feature_model(batch, training=False)
        pooled = model.get_layer("global_average_pooling")(features)
        dropped = model.get_layer("classifier_dropout")(pooled, training=False)
        score = model.get_layer("crack_probability")(dropped)[0, 0]
    gradients = tape.gradient(score, conv)
    weights = tf.reduce_mean(gradients, axis=(1, 2), keepdims=True)
    heatmap = tf.nn.relu(tf.reduce_sum(weights * conv, axis=-1))[0].numpy()
    return heatmap / heatmap.max() if heatmap.max() > 1e-12 else np.zeros_like(heatmap)

def overlay(image, heatmap):
    heat = Image.fromarray(np.uint8(np.clip(heatmap, 0, 1) * 255)).resize(image.size)
    colour = plt.get_cmap("turbo")(np.asarray(heat) / 255.0)[..., :3]
    base = np.asarray(image, dtype=np.float32) / 255.0
    return Image.fromarray(np.uint8(np.clip(0.6 * base + 0.4 * colour, 0, 1) * 255))

st.set_page_config(page_title="CrackSpot", page_icon="🔎", layout="wide")
st.title("CrackSpot — Phân loại vết nứt bề mặt")
st.caption("Crack=1, Non-crack=0; Grad-CAM chỉ là vùng mô hình chú ý.")
uploaded = st.file_uploader("Chọn ảnh JPG/PNG (tối đa 10 MB)", type=["jpg", "jpeg", "png"])
if uploaded is not None:
    if uploaded.size > 10 * 1024**2:
        st.error("Ảnh vượt giới hạn 10 MB.")
        st.stop()
    try:
        image, batch = preprocess(Image.open(uploaded))
        model, threshold = load_assets()
        started = time.perf_counter()
        probability = float(model.predict(batch, verbose=0)[0, 0])
        latency = time.perf_counter() - started
        label = "Crack" if probability >= threshold else "Non-crack"
        left, right = st.columns(2)
        left.image(image, caption="Ảnh gốc", use_container_width=True)
        right.image(overlay(image, gradcam(model, batch)), caption="Grad-CAM", use_container_width=True)
        st.metric("Kết quả", label)
        st.write({"P(Crack)": probability, "threshold": threshold, "latency_seconds": latency})
        st.warning("Kết quả hỗ trợ khảo sát sơ bộ, không thay thế đánh giá của kỹ sư.")
    except Exception as exc:
        st.error(f"Không thể xử lý ảnh: {exc}")
"""

if DO_STREAMLIT_DEMO:
    if not THRESHOLD_PATH.is_file():
        raise FileNotFoundError("Chưa có model E4 và threshold E5.")
    runtime_app = ARTIFACT_ROOT / "streamlit_app_runtime.py"
    runtime_app.write_text(STREAMLIT_APP_SOURCE, encoding="utf-8")
    environment = os.environ.copy()
    environment["CRACKSPOT_MODEL_PATH"] = str(RUN_DIRS["E4"] / "model.keras")
    environment["CRACKSPOT_THRESHOLD_PATH"] = str(THRESHOLD_PATH)
    process = subprocess.Popen(
        [PYTHON, "-m", "streamlit", "run", str(runtime_app), "--server.port", "8501"],
        env=environment,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.STDOUT,
    )
    print("Streamlit PID:", process.pid, "— http://localhost:8501")
    if IN_COLAB:
        time.sleep(3)
        from google.colab import output

        output.serve_kernel_port_as_window(8501)

## 11. Tổng hợp report facts và đóng gói artifact


In [ ]:
if DO_GENERATE_REPORT_FACTS:
    facts = {
        "schema_version": 1,
        "generated_from_notebook": True,
        "git_commit": git_commit,
        "manifest_sha256": EXPECTED_MANIFEST_SHA,
        "split_summary": split_summary.to_dict(orient="records"),
        "experiments": {},
        "benchmark": None,
        "real_images": None,
        "limitations": [
            "SDNET2018 không đại diện mọi camera, ánh sáng và vật liệu.",
            "Mất cân bằng lớp khiến Accuracy cần đọc cùng Recall/F1 Crack.",
            "Grad-CAM là heatmap thô, không phải mask hoặc bounding box.",
            "Hệ thống hỗ trợ khảo sát sơ bộ, không thay đánh giá kỹ sư.",
        ],
    }
    for target in ("e1", "e2", "e3", "e5"):
        path = REPORT_ROOT / "final_evaluation" / target / "metrics_test.json"
        if path.is_file():
            facts["experiments"][target.upper()] = json.loads(path.read_text(encoding="utf-8"))
    benchmark_path = REPORT_ROOT / "benchmark.json"
    if benchmark_path.is_file():
        facts["benchmark"] = json.loads(benchmark_path.read_text(encoding="utf-8"))
    real_path = REPORT_ROOT / "real_images/metrics_real.json"
    if real_path.is_file():
        facts["real_images"] = json.loads(real_path.read_text(encoding="utf-8"))

    required_targets = {"E1", "E2", "E3", "E5"}
    facts["submission_evidence_complete"] = (
        required_targets.issubset(facts["experiments"])
        and facts["benchmark"] is not None
        and facts["real_images"] is not None
    )
    facts_path = REPORT_ROOT / "report_facts.json"
    write_json(facts_path, facts)
    display(
        pd.DataFrame(
            [
                {
                    "experiment": name,
                    "threshold": item["threshold"],
                    "accuracy": item["metrics"]["accuracy"],
                    "precision_crack": item["metrics"]["crack"]["precision"],
                    "recall_crack": item["metrics"]["crack"]["recall"],
                    "f1_crack": item["metrics"]["crack"]["f1"],
                    "accuracy_0_92": item["accuracy_target_0_92_met"],
                }
                for name, item in facts["experiments"].items()
            ]
        )
    )
    print("Report facts:", facts_path)
    print("Submission evidence complete:", facts["submission_evidence_complete"])

if DO_ZIP_ARTIFACTS:
    archive_base = PROJECT_ROOT / "crackspot_notebook_artifacts"
    archive = shutil.make_archive(str(archive_base), "zip", ARTIFACT_ROOT)
    print("Đã đóng gói:", archive)
    if IN_COLAB:
        from google.colab import files

        files.download(archive)

## 12. Checklist trước khi nộp

- [ ] E1, E2, E3 chạy trên cùng locked split và có history/model/metadata thật.
- [ ] model_selection.json chứng minh E4 chỉ chọn bằng validation loss.
- [ ] E4 chạy augmentation và kế thừa đúng boundary/LR của winner.
- [ ] E5 tune threshold chỉ từ predictions_validation.csv.
- [ ] Final test E1/E2/E3/E5 chỉ mở một lần; registry và hash còn nguyên.
- [ ] Bảng Accuracy, Precision, Recall, F1 Crack, macro F1, confusion matrix, FP/FN lấy từ metrics_test.json.
- [ ] Có learning curves và Grad-CAM TP/TN/FP/FN thật.
- [ ] Có benchmark p95 và ảnh nhóm tự chụp đánh giá riêng.
- [ ] Notebook upload và Streamlit cùng trả P(Crack), threshold, latency và Grad-CAM.
- [ ] Accuracy ≥ 0,92 chỉ ghi là đạt nếu artifact test chứng minh, nêu rõ threshold.
- [ ] Báo cáo 35–45 trang và slide dùng report_facts.json; không dùng output giả/smoke.
- [ ] Ghi hạn chế, domain shift và cảnh báo không thay đánh giá kỹ sư.

Notebook chứa đủ code, nhưng kết quả thực nghiệm, ảnh tự chụp, báo cáo và slide vẫn phải được chạy/tạo thật trước khi nộp.
